# Tutorial 10: Reinforcement Learning Tutorial
# Part B: RL Controller Training (Soft-Actor Critic)

# Quick Recap of the Control Problem for RL Controller

- **3** Optimization Objectives<br>
- **6** Continuous Actions<br>
- **13** Observations for each step<br>

We need to implement the RL controller to replace:<br>
**action = fixed_action**<br>
with:<br>
**action = policy(obs)**<br>

# Step 1 - Imports & Model Path

In [50]:
import os
import sys
from pathlib import Path

# Project root = parent folder of the current working directory
PROJECT_ROOT = Path.cwd().resolve().parent

# Add project root to Python path so we can import from `src/`
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from stable_baselines3 import SAC  # RL algorithm (Soft Actor-Critic) from SB3
from stable_baselines3.common.utils import set_random_seed  # For reproducibility

# TensorBoard command (run in terminal) to view training logs
# tensorboard --logdir "C:\Users\Administrator\Desktop\code\MuFlex_tutorial\Tutorial_MuFlex\RL_Log\tb"

# MuFlex environment (project code)
from src.env_wrapper import MuFlex


# Step 2 - Create Environment for Interacting with SB3

In [51]:
def make_env(fmu_configs, seed):
    # Build a callable that creates a fresh MuFlex env (needed by VecEnv / SB3)
    def _init():
        env = MuFlex(
            fmu_configs=fmu_configs,          # FMU configs (which FMUs to load + I/O type)
            sim_days=1,                       # simulate for 1 day
            start_date=201,                   # Day-of-Year 201 (Jul 20 in a non-leap year)
            step_size=900,                    # Simulation step size (seconds) = 15 minutes
            action_type="continuous",         # Continuous action space (e.g., setpoints)
            include_hour=False,               # Do not include hour-of-day in observation
            reward_mode="default",            # Use the default reward function
            max_total_hvac_power=9000.0,      # Power cap used by the environment/reward
            save_results=False,               # Do not save episode outputs during training
            print_step_info=False             # Do not print simulation data (too many episodes)
        )
        return env

    # Set random seed for reproducible runs (NumPy, Python, etc.)
    set_random_seed(seed)

    # Return the env constructor (SB3 will call it to create env instances)
    return _init


# Step 3 - Paths Setup

In [52]:
# Project root = parent folder of the current working directory
project_root = Path.cwd().resolve().parent

# FMU file path
fmu_path = project_root / "models" / "small_office" / "small_baseline_v1.fmu"

# Main log folder for this RL example
log_dir = project_root / "Tutorial_MuFlex" / "RL_Log"
os.makedirs(log_dir, exist_ok=True)  # Create folder if it does not exist

# TensorBoard log folder
tb_log_dir = log_dir / "tb"
os.makedirs(tb_log_dir, exist_ok=True)  # Create folder if it does not exist

# Print paths for checking
print("project_root:", project_root)
print("fmu_path:", fmu_path)
print("log_dir:", log_dir)
print("tb_log_dir:", tb_log_dir)

project_root: C:\Users\Administrator\Desktop\code\MuFlex_tutorial
fmu_path: C:\Users\Administrator\Desktop\code\MuFlex_tutorial\models\small_office\small_baseline_v1.fmu
log_dir: C:\Users\Administrator\Desktop\code\MuFlex_tutorial\Tutorial_MuFlex\RL_Log
tb_log_dir: C:\Users\Administrator\Desktop\code\MuFlex_tutorial\Tutorial_MuFlex\RL_Log\tb


# Step 4 - Initialize Environment and Check Spaces

In [53]:
# Define which FMU to load and which I/O template to use
fmu_configs = [{"io_type": "OfficeS", "path": str(fmu_path)}]

# Create the environment (make_env returns a constructor, so we call it with () )
env = make_env(fmu_configs, seed=42)()

# Print action/observation spaces for checking
print("action_space:", env.action_space)
print("observation_space:", env.observation_space)

action_space: Box(-1.0, 1.0, (6,), float32)
observation_space: Box([  0. -30.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.], [1.0e+02 5.0e+01 1.5e+03 2.0e+04 2.0e+03 5.0e+00 5.0e+01 5.0e+01 5.0e+01
 5.0e+01 5.0e+01 5.0e+01 1.5e+01], (13,), float32)


# Step 5 - Create SAC model (Stable-Baselines3)

In [54]:
model = SAC(
    "MlpPolicy",                     # Use an MLP (fully-connected) policy network
    env,                             # Training environment
    learning_rate=3e-4,              # Optimizer learning rate
    buffer_size=200_000,             # Replay buffer size
    batch_size=256,                  # Mini-batch size for updates
    tau=0.005,                       # Soft update coefficient for target networks
    gamma=0.99,                      # Discount factor
    train_freq=1,                    # Train every N steps
    gradient_steps=1,                # Gradient updates per training step
    verbose=1,                       # Print training logs
    seed=42,                         # Random seed
    tensorboard_log=str(tb_log_dir), # TensorBoard log folder
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


# Step 6 - Train SAC Model

In [55]:
model.learn(
    total_timesteps=120000,        # Total training steps (timesteps)(total 1250 days)
    log_interval=5,                # Print logs every N updates
    tb_log_name="SAC_small_office" # TensorBoard run name (shown under tb_log_dir)
)

 [Init] Loading FMU #1: io_type = OfficeS, path = C:\Users\Administrator\Desktop\code\MuFlex_tutorial\models\small_office\small_baseline_v1.fmu 
 [Init] Initializing FMU #1, time range = (17280000, 17366400) 
Logging to C:\Users\Administrator\Desktop\code\MuFlex_tutorial\Tutorial_MuFlex\RL_Log\tb\SAC_small_office_8
 [Reward] HVAC penalty: 1.4092237413194444 
 [Reward] Temperature penalty: 0.0 
 [Reward] Max power penalty: 7.94364621239349 
 [Reward] Total penalty: 16.591904295446703 
 [Reward] Final Reward: -16.591904295446703 
 [Close] Terminating all FMUs... 
 [Close] FMU #1 terminated successfully. 
 [Close] All FMUs have been processed. 
 [Init] Loading FMU #1: io_type = OfficeS, path = C:\Users\Administrator\Desktop\code\MuFlex_tutorial\models\small_office\small_baseline_v1.fmu 
 [Init] Initializing FMU #1, time range = (17280000, 17366400) 
 [Reward] HVAC penalty: 1.0371549479166666 
 [Reward] Temperature penalty: 0.0 
 [Reward] Max power penalty: 4.302761543952093 
 [Reward] Tot

# Step 7 - Save SAC Model & Clean Up

In [56]:
model.save(log_dir / "SAC_small_office")  
# Save the trained SB3 agent (policy/critic network weights + model config)
# SB3 will save as SAC_small_office.zip
env.close()  # Close the environment and release resources

 [Close] Terminating all FMUs... 
 [Close] FMU #1 terminated successfully. 
 [Close] All FMUs have been processed. 
